# Customer Churn Prediction API Deployment

This notebook demonstrates how to:

1. Train and save a machine learning model
2. Create a FastAPI-based inference API
3. Test the API locally
4. Containerize the application using Docker

In [1]:
!pip install fastapi uvicorn pyngrok joblib

# Model Training

We train a Logistic Regression model on sample customer data and save it for deployment.

In [2]:
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
import joblib

# Generate sample dataset
X, y = make_classification(
    n_samples=1000,
    n_features=4,
    random_state=42
)

# Train model
model = LogisticRegression()
model.fit(X, y)

# Save model
joblib.dump(model, "model.pkl")

print("Model saved successfully")

Model saved successfully


In [3]:
import os

print(os.listdir())

['.config', 'model.pkl', 'sample_data']


# FastAPI Application

The API loads the trained model and exposes a prediction endpoint.

In [4]:
%%writefile app.py

from fastapi import FastAPI
from pydantic import BaseModel
import joblib
import numpy as np

app = FastAPI()

model = joblib.load("model.pkl")

class CustomerData(BaseModel):
    feature1: float
    feature2: float
    feature3: float
    feature4: float

@app.get("/")
def home():
    return {"message":"Customer Churn API"}

@app.post("/predict")
def predict(data: CustomerData):

    features = np.array([[
        data.feature1,
        data.feature2,
        data.feature3,
        data.feature4
    ]])

    prediction = model.predict(features)[0]

    return {
        "prediction": int(prediction)
    }

Writing app.py


In [5]:
!cat app.py


from fastapi import FastAPI
from pydantic import BaseModel
import joblib
import numpy as np

app = FastAPI()

model = joblib.load("model.pkl")

class CustomerData(BaseModel):
    feature1: float
    feature2: float
    feature3: float
    feature4: float

@app.get("/")
def home():
    return {"message":"Customer Churn API"}

@app.post("/predict")
def predict(data: CustomerData):

    features = np.array([[
        data.feature1,
        data.feature2,
        data.feature3,
        data.feature4
    ]])

    prediction = model.predict(features)[0]

    return {
        "prediction": int(prediction)
    }


In [6]:
!nohup uvicorn app:app --host 0.0.0.0 --port 8000 &

nohup: appending output to 'nohup.out'


In [8]:
from pyngrok import ngrok

public_url = ngrok.connect(8000)

print(public_url)

ERROR:pyngrok.process.ngrok:t=2026-06-22T11:25:10+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:pyngrok.process.ngrok:t=2026-06-22T11:25:10+0000 lvl=eror msg="session closing" obj=tunnels.session err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:pyngrok.process.ngrok:t=2026-06-22T11:25:10+0000 lvl=eror msg="terminating with error" obj=app err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your aut

PyngrokNgrokError: The ngrok process errored on start: authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n.

In [9]:
from pyngrok import ngrok

ngrok.set_auth_token("3FUROICphOIjyoVeKFnq4VwOG6t_4hzvYendRS1xruzCXt3Qi")

In [10]:
from pyngrok import ngrok

public_url = ngrok.connect(8000)

print(public_url)

NgrokTunnel: "https://earpiece-irritably-provoking.ngrok-free.dev" -> "http://localhost:8000"
